<a href="https://colab.research.google.com/github/Zain506/MedCLIP-SAM/blob/main/notebooks/ImageSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MedCLIP-SAM
[Paper](https://arxiv.org/pdf/2403.20253)
---
> With a fine-tuned BiomedCLIP model, we proposed a zero-shot universal medical image segmentation strategy, which leverages the recent XAI technique,
gScoreCAM that provides visual saliency maps of text prompts in corresponding images for CLIP models. While gScoreCAM was shown to outperform
gradCAM in natural images in accuracy and specificity, we adopted it in radiological tasks for the first time. Here, for an input image and a text prompt for
the target anatomy/pathology, we first obtained an initial, coarse segmentation
by post-processing the gScoreCAM map with a conditional random field (CRF)
filter, which was then used to obtain a bounding box for SAM to produce a
pseudo-mask as zero-shot segmentation. In the attempt to further enhance the
accuracy of zero-shot segmentation, we used the resulting pseudo-masks to train
a Residual UNet in a weakly supervised setting.

# gScoreCAM
[Paper](https://www.google.com/url?q=https%3A%2F%2Fopenaccess.thecvf.com%2Fcontent%2FACCV2022%2Fpapers%2FChen_gScoreCAM_What_objects_is_CLIP_looking_at_ACCV_2022_paper.pdf)

---

> CAM  is applied to all N channels
N at the last convolutional layer
(e.g. layer4 at ResNet-50 ) that is followed by a GAP layer and then a linear
1000-output classification layer (whose weight matrix ∈ R
N×1000). That is, for
each ImageNet class c, CAM uses the N corresponding weights {w
c
i
}
N to linearly
combine N channels to create a saliency map Mc
CAM:
Mc
CAM =
X
N
i
Ai × sof tmax(w
c
i
) (1)
ScoreCAM  is the same as CAM but uses the confidence scores of the CNN
in place of the weights w
c
i
in Eq. 1. Specifically, to explain why an input image
x belongs to a target class c w.r.t. a CNN fc(.), the ScoreCAM algorithm is:
1. Upsample all N channels at the last convolutional layer to the input-image
size using bilinear interpolation, yielding a set of upsampled channels {A
up
i
}
N ,
which serve as masks in the next step.
2. Element-wise multiply each mask A
up
i with all color channels of the input
image x and feed the resultant (masked) images to the CNN fc(.) to obtain
an output confidence score corresponding to the target class c.
3. Use the N confidence scores obtained in place of the w
c
i
to compute Mc
CAM
following Eq. 1.
In sum, the ScoreCAM saliency map is computed by:
Mc
ScoreCAM =
X
N
i
Ai × sof tmax(fc(x ⊙ A
up
i
)) (2)
where ⊙ is the Hadamard product
---
# Summary

- The segmentation process requires producing a suitable saliency map by generating a bounding box.

- We opt for a self-supervised approach of data augmentation, and then computing its similarity with the text embedding to identify the latent feature(s) most significant in encoding the image.
- We can do this because CLIP encodes the multimodal data into a shared embedding space, so a report diagnosing an image would have the same (or very similar) embedding as the image.

- gScoreCAM augments the data with every mapping in the final convolutional layer of a CNN to identify the convolution that is the most significant. This convolution mapping is used to generate the mask.

- While MedCLIP uses a Vision Transformer, the final layer patch embeddings encode its context: they are latent features like convolutions. By augmenting the input data with each patch embedding, we can identify the patch most significant to encoding the image, and use that as a bounding box.

- We post-process the saliency map to extract bounding-boxes around high activation regions.

- We feed this bounding box, and the original image, into SAM for it to highlight the region.

- SAM's mask decoder uses these visual prompts to produce a refined segmentation mask.

- We do not feed the text to SAM. All text information is used to identify the best convolution mappings to generate a bounding box via gScoreCAM

In [ ]:
%pip install open-clip-torch -q

In [ ]:
# Import data
from datasets import load_dataset
ds = load_dataset("adishourya/MEDPIX-ClinQA") # Same MedPIX dataset that fine-tuned MedCLIP
train_valid = ds["train"].train_test_split(test_size=0.1)
training = train_valid["train"].select(range(100))
test = train_valid["test"]

In [ ]:
print(training)

In [ ]:
from google.colab import drive

drive.mount("/content/drive/")

## Load BiomedCLIP

In [ ]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
weights_path = "/content/drive/MyDrive/colab/MedCLIP-SAM/biomedclip_weights.pth"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

In [ ]:
def collate_fn(batch): # Convert batch into tensor
  images = torch.stack([preprocess(x["image_id"]) for x in batch])
  texts = tokenizer([x["answer"] for x in batch])
  return images, texts

data_loader = DataLoader(
    training,
    batch_size=10,
    shuffle=True,
    collate_fn=collate_fn,
)


In [ ]:
epochs = 1
for epoch in range(epochs):
  for images, texts in tqdm(data_loader, desc=f"Epoch {epoch+1}"):
    images = images.to(device)
    texts = texts.to(device)
    I_vec = model.encode_image(images) # noqa: N806
    T = model.encode_text(texts)
    break

In [ ]:
print(I_vec.shape)